In [ ]:
from pathlib import Path


ROOT_DIR = Path.home() / "workspace"
DATA_DIR = ROOT_DIR / "dataset" / "DCASE2026-Task4"

In [ ]:
path = "/home/octoopt/workspace/dataset/DCASE2026-Task4/DCASE2026Task4Dataset.z02"

In [ ]:
import py7zr

with py7zr.SevenZipFile(path, mode="r") as z:
    z.extractall(path="./output")

In [ ]:
import subprocess
from pathlib import Path

folder = Path("/home/octoopt/workspace/dataset/DCASE2026-Task4")
parts = sorted(folder.glob("*.z*"))  # z01, z02, ...

# Combine
with open(folder / "combined.zip", "wb") as out:
    for part in parts:
        print(f"Merging {part.name}")
        out.write(part.read_bytes())

# Extract
print("Extracting...")
subprocess.run(["unzip", str(folder / "combined.zip"), "-d", str(folder)])

In [ ]:
# parts = sorted(folder.glob("*.z[0-9][0-9]"))  # only z01, z02, ...


In [ ]:
import subprocess
from pathlib import Path


def extract_split_archive(folder: Path, first_part: str = None):
    # Find first part
    if first_part:
        first = folder / first_part
    else:
        parts = sorted(folder.glob("*.z01"))
        if not parts:
            raise FileNotFoundError("No .z01 file found")
        first = parts[0]

    # List all parts
    all_parts = sorted(p.name for p in folder.glob("*.z*")) + sorted(
        p.name for p in folder.glob("*.zip")
    )
    print(f"First part: {first.name}")
    print(f"All parts found: {all_parts}")

    # Check all sequential parts exist
    z_parts = sorted(folder.glob("*.z[0-9]*"))
    for i, part in enumerate(z_parts, start=1):
        expected = first.stem + f".z{i:02d}"
        if part.name != expected:
            print(f"Warning: expected {expected}, found {part.name}")

    # Check 7z installed
    result = subprocess.run(["which", "7z"], capture_output=True)
    if result.returncode != 0:
        print("Installing p7zip-full...")
        subprocess.run(["sudo", "apt", "install", "-y", "p7zip-full"], check=True)

    # Extract
    output_dir = folder / "extracted"
    output_dir.mkdir(exist_ok=True)
    print(f"Extracting to {output_dir}...")

    subprocess.run(
        ["7z", "x", str(first), f"-o{output_dir}", "-y"],
        check=True,
    )
    print("Done.")
    return output_dir


if __name__ == "__main__":
    folder = Path("/home/octoopt/workspace/dataset/DCASE2026-Task4")
    extract_split_archive(folder)